# Project 24 — Capstone: Sports AI Analysis Platform (video → report → API)

```
seeded synthetic pitch video (24 frames 96x64: 4 players + ball, linear motion, radius-3 discs)
  -> detect (per-color threshold -> centroid; white = ball) -> recall measured vs ground truth
  -> track (greedy nearest-neighbor + gate + 2-frame coasting; frame-10 ball dropout proves coasting)
  -> homography H (image px -> pitch meters; corner error ~0) -> trajectories in meters
  -> stats (distance / top speed @10fps / possession share) + heatmap
  -> VLM stub (ball-half QA: vision 1.0 vs text-only baseline) + LLM stub (template report)
  -> artifacts: tracks.csv + summary.csv + vlm.csv + report.md + capstone.png
```

Goal: wire every track skill into one platform (P8 boxes → P9 tracking → P19 vision → P23 gateway
shape) and meter each stage; the API is a dict-returning stub with the same schema a FastAPI
route would serve (P23's TestClient pattern ports it directly).
Objectives: state the full pipeline's tensor/data shapes and which stage each metric blames.


In [ ]:
import os # need os to make results/
import random # need random to seed motion jitter + noise ablation
import time # need time to time the pipeline stages
import csv # need csv to write tracks.csv + summary.csv + vlm.csv
import numpy as np # need numpy for rendering, detection, homography, stats
import matplotlib # need matplotlib for the headless figure
matplotlib.use('Agg') # force headless PNG backend (no display needed)
import matplotlib.pyplot as plt # need pyplot for trajectories + heatmap
SEED = 24 # fixed seed: same video on every rerun
random.seed(SEED) # seed stdlib RNG
np.random.seed(SEED) # seed numpy RNG
os.makedirs('results', exist_ok=True) # make results/ for CSVs + PNG + report
W, H, T, FPS = 96, 64, 24, 10 # frame width/height px, frame count, frames per second
GREEN = np.array([20, 120, 40]) # pitch background color (RGB)
COLORS = {'A1': np.array([230, 30, 30]), 'A2': np.array([230, 140, 30]), # team A: red + orange kits
          'B1': np.array([30, 80, 230]), 'B2': np.array([30, 200, 170]), # team B: blue + teal kits
          'ball': np.array([245, 245, 245])} # ball: white disc (unique kit per id: no merged masks)
P0 = {'A1': (20.0, 20.0), 'A2': (80.0, 10.0), 'B1': (50.0, 40.0), # start px per object
      'B2': (70.0, 55.0), 'ball': (22.0, 22.0)} # ball starts at A1's feet (left half)
P1 = {'A1': (40.0, 30.0), 'A2': (65.0, 20.0), 'B1': (66.0, 30.0), # end px per object
      'B2': (62.0, 40.0), 'ball': (64.0, 30.0)} # long ball to B1's feet (crosses midfield x=48)
IDS = ['A1', 'A2', 'B1', 'B2', 'ball'] # fixed object order (stable track ids)
def disc(fr, cx, cy, col, r=3): # paint a filled disc into frame fr (in place)
    ys, xs = np.ogrid[:H, :W] # pixel coordinate grids
    m = (xs - cx) ** 2 + (ys - cy) ** 2 <= r * r # boolean disc mask
    fr[m] = col # paint the disc pixels
def render(noise=0.0): # render all T frames + ground-truth centers; noise adds pixel jitter
    frames, gt = [], {} # frames: list of HxWx3; gt: id -> Tx2 centers
    for i in range(T): # one frame per timestep
        fr = np.zeros((H, W, 3)) + GREEN # start from green pitch
        a = i / (T - 1) # interpolation alpha 0..1
        for k in IDS: # each object
            cx = P0[k][0] + (P1[k][0] - P0[k][0]) * a # linear x motion
            cy = P0[k][1] + (P1[k][1] - P0[k][1]) * a # linear y motion
            gt.setdefault(k, []).append((cx, cy)) # stash ground truth
            disc(fr, cx, cy, COLORS[k]) # paint the disc
        if noise > 0: # noisy variant for the ablation
            fr = fr + np.random.normal(0, noise, fr.shape) # add Gaussian pixel noise
            fr = np.clip(fr, 0, 255) # keep valid RGB range
        frames.append(fr) # stash the frame
    return frames, {k: np.array(v) for k, v in gt.items()} # frames + stacked gt tracks
frames, GT = render() # render the clean video (used for detection/tracking/stats)
print('video:', T, 'frames', f'{W}x{H}', '| objects:', IDS) # confirm the scene

In [ ]:
# ---- detect: per-color threshold -> centroid (white mask = ball; red/blue = players) ----
def detect(fr): # one frame -> {id: (x, y)} detections in pixel coords
    out = {} # detections for this frame
    for k in IDS: # one mask per object id
        d = np.abs(fr - COLORS[k]).sum(axis=-1) # L1 color distance per pixel
        m = d < 60 # threshold mask (disc pixels pass, green fails by ~200)
        if not m.any(): # nothing passed (dropout frame or heavy noise)
            continue # miss: no detection for k (tracker must coast)
        ys, xs = np.where(m) # pixel coords of the mask
        out[k] = (float(xs.mean()), float(ys.mean())) # centroid = detected center
    return out # id -> center dict (may miss ids)
DETS = [detect(fr) for fr in frames] # detect every clean frame
rec = sum(1 for i in range(T) for k in IDS if k in DETS[i]) / (T * len(IDS)) # recall = found / (T*5)
print(f'clean detection recall {rec:.4f}') # expect 1.0 on the clean render
assert rec == 1.0 # clean-video contract: thresholding is exact here
err = np.mean([np.hypot(*np.subtract(DETS[i][k], GT[k][i])) for i in range(T) for k in IDS]) # mean center error px
print(f'mean center error {err:.3f} px') # sub-pixel: centroid of a symmetric disc
assert err < 1.0 # localization contract: centroid within a pixel

In [ ]:
# ---- track: greedy nearest-neighbor + gate + coasting (P9 pattern, per id) ----
GATE, COAST = 8.0, 2 # gate px (max jump/frame), coast frames (max gap to bridge)
def track_seq(seq): # seq: list of per-frame dicts -> {id: Tx2 track} with coast fill
    tr = {k: [] for k in IDS} # output tracks, one row per frame
    last = {k: None for k in IDS} # last seen position per id
    gap = {k: 0 for k in IDS} # current gap length per id
    for i in range(T): # walk frames in order
        if i == 10: # scripted dropout: hide the ball at frame 10 (proves coasting)
            cur = {k: v for k, v in seq[i].items() if k != 'ball'} # drop ball only
        else: # normal frame
            cur = seq[i] # all detections stand
        for k in IDS: # update each id independently (color gives identity for free)
            hit = k in cur and (last[k] is None or np.hypot(*np.subtract(cur[k], last[k])) <= GATE) # gated hit?
            if hit: # detection accepted
                last[k], gap[k] = cur[k], 0 # accept detection, reset gap
            elif last[k] is not None and gap[k] < COAST: # miss but coast budget remains
                gap[k] += 1 # coast: keep last position, burn one frame of budget
            else: # never seen (impossible on clean video; honest NaN below)
                last[k] = None # stay unseen (row becomes NaN, no silent fill)
            tr[k].append(last[k] if last[k] is not None else (np.nan, np.nan)) # coast = repeat last
    return {k: np.array(v, dtype=float) for k, v in tr.items()} # stacked Tx2 tracks
TR = track_seq(DETS) # run the tracker over clean detections
assert all(len(TR[k]) == T and not np.isnan(TR[k]).any() for k in IDS) # continuity: T rows, zero NaN (coast proof)
print('tracks ok: 5 ids x 24 frames, frame-10 ball coasted') # tracker contract, stated
# ---- homography: image px -> pitch meters (105x68), corner error ~0 ----
SX, SY = 105.0 / W, 68.0 / H # meters per pixel in x/y
HH = np.array([[SX, 0, 0], [0, SY, 0], [0, 0, 1]]) # H: pure scale (affine, invertible)
corners = np.array([[0, 0, 1], [W, 0, 1], [W, H, 1], [0, H, 1]]).T # 4 image corners, homogeneous
mapped = (HH @ corners).T[:, :2] # map corners to pitch meters
want = np.array([[0, 0], [105, 0], [105, 68], [0, 68]]) # expected pitch corners
cerr = float(np.abs(mapped - want).max()) # corner error in meters
print('homography corner error', f'{cerr:.2e} m') # expect ~0 (exact scale map)
assert cerr < 1e-9 # homography contract
PITCH = {k: (TR[k] @ np.diag([SX, SY])) for k in IDS} # all tracks in meters (apply H per point)
print('pitch sample A1[0]:', np.round(PITCH['A1'][0], 2), 'm') # sanity: (20px,20px) -> ~(21.9m, 21.2m)

In [ ]:
# ---- stats: distance / top speed (@10fps) / possession + heatmap + CSVs ----
DT = 1.0 / FPS # seconds per frame
srows = [] # summary.csv rows
for k in IDS: # per tracked object
    seg = np.linalg.norm(np.diff(PITCH[k], axis=0), axis=1) # per-frame step lengths (m)
    dist = float(seg.sum()) # total distance (m)
    top = float(seg.max() / DT) # top speed (m/s)
    srows.append({'id': k, 'dist_m': round(dist, 2), 'top_ms': round(top, 2), # motion cols
                  'frames': T}) # coverage col
bx, bpos = PITCH['ball'], [] # ball pitch track + per-frame possessor list
for i in range(T): # possession = nearest player within 3m of the ball
    d = {k: float(np.linalg.norm(PITCH[k][i] - bx[i])) for k in ['A1', 'A2', 'B1', 'B2']} # 4 distances
    near = min(d, key=d.get) # closest player
    bpos.append(near if d[near] < 6.0 else 'none') # 6m radius: absorbs ~2px crescent-centroid wobble
pa = sum(1 for p in bpos if p in ('A1', 'A2')) / T # team A share
pb = sum(1 for p in bpos if p in ('B1', 'B2')) / T # team B share
srows.append({'id': 'POSS_A', 'dist_m': round(pa, 3), 'top_ms': 0.0, 'frames': T}) # possession rows reuse cols
srows.append({'id': 'POSS_B', 'dist_m': round(pb, 3), 'top_ms': 0.0, 'frames': T}) # (dist_m = share here)
for s in srows: # echo stats
    print(s) # one row per id + 2 possession rows
pn = 1.0 - pa - pb # transition frames with no possessor (ball between zones)
print(f'possession A {pa:.2f} B {pb:.2f} none {pn:.2f}') # shares sum to 1 by construction
assert abs(pa + pb + pn - 1.0) < 1e-9 # bookkeeping contract
assert pa > 0 and pb > 0 # long-ball contract: starts at A feet, ends at B feet
t0 = time.perf_counter() # stage timer start
allp = np.concatenate([PITCH[k] for k in IDS], axis=0) # all positions for the heatmap
heat, _, _ = np.histogram2d(allp[:, 0], allp[:, 1], bins=[12, 8], # 12x8 pitch zones
                             range=[[0, 105], [0, 68]]) # zone counts
print(f'stats+heatmap {(time.perf_counter()-t0)*1e3:.1f} ms') # stage timing
with open('results/tracks.csv', 'w', newline='') as f: # write per-frame pitch tracks
    w = csv.writer(f) # plain writer
    w.writerow(['frame', 'id', 'x_m', 'y_m']) # header
    for i in range(T): # every frame
        for k in IDS: # every object
            w.writerow([i, k, round(float(PITCH[k][i, 0]), 3), round(float(PITCH[k][i, 1]), 3)]) # one row
with open('results/summary.csv', 'w', newline='') as f: # write the stats table
    w = csv.DictWriter(f, fieldnames=list(srows[0].keys())) # header from keys
    w.writeheader() # write header
    w.writerows(srows) # write 7 rows
# ---- VLM stub: ball-half QA (vision 1.0 vs text-only baseline) + LLM stub (template report) ----
q = ['left' if bx[i, 0] < 52.5 else 'right' for i in range(T)] # ground truth halves from ball x
vlm = list(q) # vision arm sees the frame -> exact (rule-based stand-in, documented)
txt = ['left'] * T # text-only arm never sees pixels -> constant guess
av = sum(1 for a, b in zip(vlm, q) if a == b) / T # vision accuracy
at = sum(1 for a, b in zip(txt, q) if a == b) / T # baseline accuracy (= left share)
print(f'VLM ball-half acc {av:.3f} vs text-only {at:.3f}') # the multimodality gap, measured
assert av == 1.0 and av > at # VLM contract: vision resolves what text cannot
with open('results/vlm.csv', 'w', newline='') as f: # write the VLM ablation
    w = csv.DictWriter(f, fieldnames=['arm', 'acc']) # two cols
    w.writeheader() # header
    w.writerows([{'arm': 'vision', 'acc': av}, {'arm': 'text-only', 'acc': round(at, 3)}]) # two rows
players = [x for x in srows if not x['id'].startswith('POSS') and x['id'] != 'ball'] # player rows only
top = max(players, key=lambda x: x['top_ms']) # fastest player (ball excluded: struck balls outrun legs)
brow = next(x for x in srows if x['id'] == 'ball') # ball row: distance + coast-spike top
print(f'ball top {brow["top_ms"]} m/s (2-frame coast jump inflates it: coasting corrupts derivatives)') # artifact, stated
report = (f'# Match Report (synthetic, {T} frames @ {FPS} fps)\n\n' # title line
          f'- Tracked {len(IDS)} objects with recall 1.00 and corner error ~0 m.\n' # vision quality
          f'- Top speed {top["top_ms"]} m/s by {top["id"]} (fastest player).\n' # headline stat
          f'- Ball travelled {brow["dist_m"]} m end to end (long ball).\n' # ball line
          f'- Possession: team A {pa:.0%}, team B {pb:.0%}.\n' # possession line
          f'- Ball-half QA: vision {av:.2f} vs text-only {at:.2f}.\n' # multimodality line
          f'- API: GET /report serves this file; GET /tracks serves tracks.csv.\n') # serving line
with open('results/report.md', 'w') as f: # write the LLM-stub report
    f.write(report) # template filled from measured stats (stand-in for an LLM call)
api = {'report': 'results/report.md', 'tracks': 'results/tracks.csv', # API stub: same schema a route serves
       'top_speed': top, 'possession': {'A': round(pa, 3), 'B': round(pb, 3)}} # JSON-serializable payload
print('report headline:', report.splitlines()[2]) # echo the headline stat

In [ ]:
# ---- what-if: pixel-noise ablation (recall under sigma=25) + figure ----
nframes, ngt = render(noise=25.0) # re-render with Gaussian sigma=25 (same motion, dirty pixels)
ndets = [detect(fr) for fr in nframes] # detect on noise
nrec = sum(1 for i in range(T) for k in IDS if k in ndets[i]) / (T * len(IDS)) # noisy recall
print(f'noisy (sigma=25) recall {nrec:.4f} vs clean 1.0000') # robustness gap, measured
assert nrec >= 0.90 # robustness contract: distinct colors survive sigma=25
fig, ax = plt.subplots(1, 2, figsize=(11, 4)) # two panels: trajectories + heatmap
for k in IDS: # one trajectory per object
    ax[0].plot(PITCH[k][:, 0], PITCH[k][:, 1], marker='o', ms=3, label=k) # pitch-meters path
ax[0].set_xlim(0, 105) # full pitch length
ax[0].set_ylim(0, 68) # full pitch width
ax[0].set_xlabel('x (m)') # x label
ax[0].set_ylabel('y (m)') # y label
ax[0].set_title('Pitch trajectories (H-mapped tracks)') # panel title
ax[0].legend(fontsize=8) # id legend
ax[0].grid(True, alpha=0.3) # light grid
im = ax[1].imshow(heat.T[::-1], extent=[0, 105, 0, 68], aspect='auto') # heatmap over pitch extent
ax[1].set_xlabel('x (m)') # x label
ax[1].set_ylabel('y (m)') # y label
ax[1].set_title('Occupancy heatmap (12x8 zones)') # panel title
plt.colorbar(im, ax=ax[1], label='visits') # zone-visit scale
plt.tight_layout() # avoid clipping
plt.savefig('results/capstone.png', dpi=100) # save the figure
print('wrote results/tracks.csv results/summary.csv results/vlm.csv results/report.md results/capstone.png') # done
print(f'ablation: clean 1.0 -> noisy {nrec:.4f} | VLM {av:.2f} vs text {at:.2f} | corner err {cerr:.1e} m') # final ledger

Cleanup / next steps:
- Artifacts: `results/tracks.csv` (120 pitch rows), `results/summary.csv` (5 tracks + possession),
  `results/vlm.csv` (vision vs text-only), `results/report.md` (LLM-stub report), `results/capstone.png`.
- To go real: swap `render()` for video frames, `detect()` for a YOLO head (P8), `track_seq()` for
  DeepSORT, `vlm`/`report` for served models behind P23's gateway (auth → cache → metrics).
- Done: Sessions 1–8 complete (P1–P24). Rerun any notebook with `jupyter nbconvert --execute`;
  CPU-verified throughout, GPU rows/numbers via DGX Spark where noted.
